# Tools in LlamaIndex


This notebook is part of the [Hugging Face Agents Course](https://www.hf.co/learn/agents-course), a free Course from beginner to expert, where you learn to build Agents.

![Agents course share](https://huggingface.co/datasets/agents-course/course-images/resolve/main/en/communication/share.png)

## Let's install the dependencies

We will install the dependencies for this unit.

In [3]:
!pip install llama-index llama-index-vector-stores-chroma llama-index-llms-huggingface-api llama-index-embeddings-huggingface llama-index-tools-google -U -q

And, let's log in to Hugging Face to use serverless Inference APIs.

In [ ]:
from huggingface_hub import login

login()

In [ ]:
!echo "token $HF_TOKEN"

## Creating a FunctionTool

Let's create a basic `FunctionTool` and call it.

In [ ]:
from llama_index.core.tools.types import ToolOutput


from llama_index.core.tools import FunctionTool


def get_weather(location: str) -> str:
    """Useful for getting the weather for a given location."""
    print(f"Getting weather for {location}")
    return f"The weather in {location} is sunny"


tool = FunctionTool.from_defaults(
    get_weather,
    name="my_weather_tool",
    description="Useful for getting the weather for a given location.",
)
resp: ToolOutput = tool.call("New York")
print(resp.content)

## Creating a QueryEngineTool

Let's now re-use the `QueryEngine` we defined in the [previous unit on tools](/tools.ipynb) and convert it into a `QueryEngineTool`. 

In [ ]:
from llama_index.llms.huggingface_api.base import HuggingFaceInferenceAPI


import chromadb

from llama_index.core import VectorStoreIndex
from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.tools import QueryEngineTool
from llama_index.vector_stores.chroma import ChromaVectorStore

import os



hf_token = os.environ.get("HF_TOKEN")

db = chromadb.PersistentClient(path="./alfred_chroma_db")
chroma_collection = db.get_or_create_collection("alfred")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
model = "meta-llama/Llama-3.1-8B-Instruct"
model = "meta-llama/Llama-3.2-3B-Instruct"
model = "deepseek-ai/DeepSeek-V3-0324"
model = "google-t5/t5-small"
model = "meta-llama/Llama-3.2-1B-Instruct"
model = "llama3.2:1b"



# llm: HuggingFaceInferenceAPI = HuggingFaceInferenceAPI(model_name=model, token=hf_token)
from llama_index.llms.ollama import Ollama
llm = Ollama(model=model, request_timeout=60.0)




llm.complete("Hello...")

In [7]:
index = VectorStoreIndex.from_vector_store(
    vector_store=vector_store, embed_model=embed_model
)
query_engine = index.as_query_engine(llm=llm)
tool = QueryEngineTool.from_defaults(
    query_engine=query_engine,
    name="some useful name",
    description="some useful description",
)


In [ ]:

resp = await tool.acall(
    "Responds about research on the impact of AI on the future of work and society?"
)

import pprint

pprint.pprint(resp.content)

## Creating Toolspecs

Let's create a `ToolSpec` from the `GmailToolSpec` from the LlamaHub and convert it to a list of tools. 

In [ ]:
from llama_index.tools.google import GmailToolSpec

tool_spec = GmailToolSpec()
tool_spec_list = tool_spec.to_tool_list()
tool_spec_list

To get a more detailed view of the tools, we can take a look at the `metadata` of each tool.

In [ ]:
[(tool.metadata.name, tool.metadata.description) for tool in tool_spec_list]

In [ ]:
import dotenv

from llama_index.llms.openai import OpenAI
from llama_index.core.agent.workflow import FunctionAgent, ToolCallResult, ToolCall
from llama_index.core.agent.workflow.function_agent import FunctionAgent
from llama_index.core.workflow import Context
from llama_index.llms.ollama import Ollama
from llama_index.tools.mcp import BasicMCPClient, McpToolSpec
from llama_index.tools.mcp.base import McpToolSpec

dotenv.load_dotenv()

SYSTEM_PROMPT = """\
You are an AI assistant.

Before you help a user, you need to fetch the ip info first, to help you follow the laws of the country.
"""

# Ollama Model
model = "llama3.2:1b"
llm = Ollama(model=model, request_timeout=60.0)
# ollama function call with llama does not work well

# For example check above public IP addresses like 172.217. 22.14 or 8.8.8.8
# llm = OpenAI(model="gpt-4o")

async def get_agent(tools: McpToolSpec):
    tools: McpToolSpec = await tools.to_tool_list_async()
    
    print("MCP Tools:")
    for tool in tools:
        print(tool.metadata.name, tool.metadata.description)
    
    agent: FunctionAgent = FunctionAgent(
        name="Agent",
        description="An agent that can fetch the ip info of the user.",
        tools=tools,
        llm=llm,
        system_prompt=SYSTEM_PROMPT,
    )
    return agent

async def handle_user_message(
    message_content: str,
    agent: FunctionAgent,
    agent_context: Context,
    verbose: bool = False,
):
    handler = agent.run(message_content, ctx=agent_context)
    async for event in handler.stream_events():
        if verbose and type(event) == ToolCall:
            print(f"Calling tool {event.tool_name} with kwargs {event.tool_kwargs}")
        elif verbose and type(event) == ToolCallResult:
            print(f"Tool {event.tool_name} returned {event.tool_output}")

    response = await handler
    return str(response)



# We consider there is a mcp server running on 127.0.0.1:8000, or you can use the mcp client to connect to your own mcp server.
mcp_client = BasicMCPClient("http://127.0.0.1:8000/sse")
mcp_tool = McpToolSpec(client=mcp_client)

# get the agent
agent = await get_agent(mcp_tool)

# create the agent context
agent_context = Context(agent)


# Run the agent!
while True:
    user_input = input("Enter your message: ")
    if user_input == "exit":
        break
    print("User: ", user_input)
    response = await handle_user_message(user_input, agent, agent_context, verbose=True)
    print("Agent: ", response)
